# 06 · Emission integration, F2, the grouped two-channel emission

Runs the F2 grid, joint-trained BKT whose emission carries the two grouped misconception chains, conceptual and procedural, as separate additive channels beside slip, every capture rate a named parameter. No post-chain scalar pooling, the five annotation families are pre-grouped into two chains by the P over A over N merge and the additive sum combines the two channels, so each group keeps its own beta into the emission and the central readout is the conceptual against procedural comparison, made on the contributions beta times state, not the raw betas, whose feature scales differ.

The chains are GroupedChains under the chain of record, strong snap, fitted once and shared frozen across every row. The channel inputs at turn k are the two filtered states before that turn, solution row consumed by the chains, never by BKT. Evaluation is paper-aligned, each dialogue's first scored turn updates the filter and is excluded from metrics, unseen KCs contribute 0.5, exactly 0.5 classifies to 0.

**The rows:**

- Validity, beta pinned to 0, the engine's like-for-like BKT refit. Expected to land within a few tenths of the frozen M1, 60.65 accuracy, 64.28 AUC, 55.60 f1, and to match notebook 05's validity row closely, the residual being optimizer and initialization, not protocol.
- The grid, three connections, mastered, unmastered, both, each under free betas and betas pinned to 1.

**Reading order.** Validity first, then the free mastered row's two betas, the split is the finding this rung exists for, then the deltas in section 4. Fitted betas read as lower bounds, attenuated by chain measurement noise. On the both rows the four betas fit independently and the mastered against unmastered split locates where capture acts.

## 1. Setup

Grouped chains fitted once, shared by every row.

In [1]:
import pandas as pd
from scripts.load_data import load_paper_filtered_data
from scripts.chain import TriggerChain
from scripts.misconception_chains_grouped import GroupedChains
from scripts.emission_integration_pooled import CHAIN_OF_RECORD
from scripts.emission_integration_grouped import (
    EmissionIntegrationGroupedMastered,
    EmissionIntegrationGroupedUnmastered,
    EmissionIntegrationGroupedBoth,
)

train_df = load_paper_filtered_data("data/mathdial_train.csv")
test_df = load_paper_filtered_data("data/mathdial_test.csv")

chains = GroupedChains(train_df, test_df, chain_class=TriggerChain,
                       chain_kwargs=dict(CHAIN_OF_RECORD))
chains.run()
chains.summary()

,chain,pi,onset,resolve,pP_i,pP_a,expected_dwell_turns,accuracy,tpr,tnr,auc,f1,informative_cells
0,conceptual,0.05,0.0,0.0348,0.02,0.97,28.7,0.6324,0.6683,0.4100,0.5801,0.7579,1439
1,procedural,0.05,0.0,0.0870,0.02,0.97,11.5,0.6739,0.5610,0.8047,0.7250,0.6487,1282


## 2. Models

Seven fits, loop-built so each row's configuration is its name.

In [2]:
CONNECTIONS = {
    "mastered": EmissionIntegrationGroupedMastered,
    "unmastered": EmissionIntegrationGroupedUnmastered,
    "both": EmissionIntegrationGroupedBoth,
}

ROWS = {"validity, beta=0": (EmissionIntegrationGroupedMastered,
                             {"pin_beta": 0.0})}
for connection, cls in CONNECTIONS.items():
    ROWS[f"{connection}, free"] = (cls, {})
    ROWS[f"{connection}, beta=1"] = (cls, {"pin_beta": 1.0})

models = {}
for name, (cls, kwargs) in ROWS.items():
    model = cls(train_df, test_df, chains=chains, **kwargs)
    model.run()
    models[name] = model
    print(f"{name:22s} {model.metrics}")

validity, beta=0       {'accuracy': 0.6035, 'auc': 0.6397, 'f1': 0.5531, 'turns': 1985, 'beta_mastered_conceptual': 0.0, 'beta_mastered_procedural': 0.0, 'beta_unmastered_conceptual': None, 'beta_unmastered_procedural': None, 'connection': 'mastered'}
mastered, free         {'accuracy': 0.606, 'auc': 0.6393, 'f1': 0.5485, 'turns': 1985, 'beta_mastered_conceptual': 0.1608, 'beta_mastered_procedural': 0.0629, 'beta_unmastered_conceptual': None, 'beta_unmastered_procedural': None, 'connection': 'mastered'}
mastered, beta=1       {'accuracy': 0.605, 'auc': 0.6417, 'f1': 0.5767, 'turns': 1985, 'beta_mastered_conceptual': 1.0, 'beta_mastered_procedural': 1.0, 'beta_unmastered_conceptual': None, 'beta_unmastered_procedural': None, 'connection': 'mastered'}
unmastered, free       {'accuracy': 0.603, 'auc': 0.6361, 'f1': 0.5879, 'turns': 1985, 'beta_mastered_conceptual': None, 'beta_mastered_procedural': None, 'beta_unmastered_conceptual': 0.51, 'beta_unmastered_procedural': 0.4429, 'connection

## 3. Results table

The frozen M1 trio is the anchor, deltas against it are context, section 4 carries the inference.

In [3]:
M1 = {"accuracy": 0.6065, "auc": 0.6428, "f1": 0.5560}

results = pd.DataFrame(
    [{"row": name, **models[name].metrics} for name in ROWS])
for metric in ("accuracy", "auc", "f1"):
    results[f"d_{metric}"] = (results[metric] - M1[metric]).round(4)
results = results.set_index("row")
results[["connection", "beta_mastered_conceptual", "beta_mastered_procedural",
         "beta_unmastered_conceptual", "beta_unmastered_procedural",
         "accuracy", "d_accuracy", "auc", "d_auc", "f1", "d_f1", "turns"]]

,connection,beta_mastered_conceptual,beta_mastered_procedural,beta_unmastered_conceptual,beta_unmastered_procedural,accuracy,d_accuracy,auc,d_auc,f1,d_f1,turns
row,,,,,,,,,,,,
"validity, beta=0",mastered,0.0000,0.0000,NaN,NaN,0.6035,-0.0030,0.6397,-0.0031,0.5531,-0.0029,1985
"mastered, free",mastered,0.1608,0.0629,NaN,NaN,0.6060,-0.0005,0.6393,-0.0035,0.5485,-0.0075,1985
"mastered, beta=1",mastered,1.0000,1.0000,NaN,NaN,0.6050,-0.0015,0.6417,-0.0011,0.5767,0.0207,1985
"unmastered, free",unmastered,NaN,NaN,0.5100,0.4429,0.6030,-0.0035,0.6361,-0.0067,0.5879,0.0319,1985
"unmastered, beta=1",unmastered,NaN,NaN,1.0000,1.0000,0.6020,-0.0045,0.6344,-0.0084,0.5789,0.0229,1985
"both, free",both,0.1084,0.0529,0.5116,0.4376,0.5935,-0.0130,0.6272,-0.0156,0.5563,0.0003,1985
"both, beta=1",both,1.0000,1.0000,1.0000,1.0000,0.5481,-0.0584,0.5943,-0.0485,0.2666,-0.2894,1985


## 4. Deltas against the engine baseline

Every model against the validity row's own three metrics, code path, protocol, and optimizer held fixed, so each delta isolates what the channel configuration changed. Sorted by AUC delta.

In [4]:
baseline = models["validity, beta=0"].metrics
deltas = pd.DataFrame([
    {"row": name,
     "d_accuracy": round(models[name].metrics["accuracy"]
                         - baseline["accuracy"], 4),
     "d_auc": round(models[name].metrics["auc"] - baseline["auc"], 4),
     "d_f1": round(models[name].metrics["f1"] - baseline["f1"], 4)}
    for name in ROWS if name != "validity, beta=0"])
deltas.set_index("row").sort_values("d_auc", ascending=False)

,d_accuracy,d_auc,d_f1
row,,,
"mastered, beta=1",0.0015,0.0020,0.0236
"mastered, free",0.0025,-0.0004,-0.0046
"unmastered, free",-0.0005,-0.0036,0.0348
"unmastered, beta=1",-0.0015,-0.0053,0.0258
"both, free",-0.0100,-0.0125,0.0032
"both, beta=1",-0.0554,-0.0454,-0.2865


## 5. Slip attribution

Each KC's fitted slip on the free mastered row against the validity row's, same optimizer both sides, so the drop is the channel's re-attribution of what the baseline filed as unexplained slip.

In [5]:
m2 = models["mastered, free"]
bkt = models["validity, beta=0"]
slips = pd.DataFrame([
    {"kc": kc, "m2_slip": round(p["slip"], 4),
     "bkt_slip": round(bkt.parameters[kc]["slip"], 4)}
    for kc, p in m2.parameters.items()])
slips["drop"] = (slips["bkt_slip"] - slips["m2_slip"]).round(4)
print(f"median slip drop {slips['drop'].median():+.4f} over {len(slips)} KCs, "
      f"beta_mastered_conceptual {m2.metrics['beta_mastered_conceptual']}, "
      f"beta_mastered_procedural {m2.metrics['beta_mastered_procedural']}")
slips.sort_values("drop", ascending=False).head(10)

median slip drop +0.0423 over 138 KCs, beta_mastered_conceptual 0.1608, beta_mastered_procedural 0.0629


,kc,m2_slip,bkt_slip,drop
1,"Add and subtract within 1000, using concrete m...",0.0001,0.5531,0.5530
19,Compare two fractions with different numerator...,0.5109,0.9999,0.4890
23,"Count to 120, starting at any number less than...",0.1869,0.4301,0.2432
65,Interpret multiplication as scaling (resizing)...,0.2582,0.5000,0.2418
97,Solve systems of linear equations exactly and ...,0.3879,0.6000,0.2121
4,Analyze and solve pairs of simultaneous linear...,0.3879,0.6000,0.2121
30,Determine the unknown whole number in a multip...,0.2187,0.4152,0.1965
51,Find whole-number quotients of whole numbers w...,0.5933,0.7850,0.1917
64,Interpret expressions that represent a quantit...,0.0763,0.2443,0.1680
96,Solve real-world and mathematical problems by ...,0.2723,0.4401,0.1678


## 6. Interpretation ledger

- The validity row's distance from M1, and from notebook 05's validity row, bounds what initialization and the optimizer contribute, read every delta net of it.
- The free mastered row's two betas are the rung's central readout, whether conceptual belief captures at a different rate than procedural, each a lower bound under chain measurement noise. Compare the contributions beta times mean state rather than the raw betas, the two groups' state distributions differ, conceptual averaging near 0.57 and procedural near 0.28, so raw betas are not on a common scale, and no sum of them is comparable to notebook 05's pooled beta, the features and combination rules differ.
- On the both rows the mastered and unmastered betas fit independently, a stable mastered pair beside drifting or bound-hitting unmastered values reads as competence-side capture, the unmastered branch's clipped likelihood absorbing variance rather than measuring.
- The pinned both row subtracts up to 2 from each branch, deep truncation, either group live asserts near-certain incorrectness, expect its f1 behavior to be the F1 pinned rows amplified.
- Grouping earns its parameters only if the free rows here beat notebook 05's corresponding free rows, that cross-notebook comparison is legitimate, identical universe, protocol, and engine, only the channel grain differs.

## 7. Why the grouped channels did not improve prediction

The two-channel grid inherits every mechanism in notebook 05's section 7, the same-turn timing gap, the dialogue-level persistence, the redundancy with correctness history, and the test-set selection and initialization caveats, controlled same-initialization restarts across three seeds put the grouped free delta between minus 0.0026 and minus 0.0003, consistently null to slightly harmful, and part of this saved run's minus 0.0023 was the since-fixed initialization confound. Three difficulties are specific to this rung.

**The grouped pre-turn states are weak correctness predictors outright.** As direct error predictors on the test turns the conceptual state manages AUC 0.564 and the procedural state 0.520, essentially chance, despite the procedural chain's strong AUC at its own task in notebook 04. The two tasks are different, chain AUC asks whether the state predicts the family's own P and A annotations, this rung asks whether the pre-turn state predicts correctness beyond BKT, and good annotation prediction does not imply incremental correctness prediction.

**The grouping merge is lossier than its design sentence suggested.** A group cell reads A when any member shows A and none shows P, and since N carries nothing, one member's clean handling stands in for the whole disjunction. On the test data that compromise is nearly the only case, 99.4 per cent of conceptual group-A rows and 91.6 per cent of procedural ones contain exactly one member A, and no conceptual test row has all three members A at once, so the grouped state's absence evidence is almost entirely single-member, weakening what the latent means and plausibly contributing to the weaker results here.

**The beta comparison needs the state scales, and survives them at contribution level only.** The raw pair, 0.185 conceptual against 0.112 procedural, is not evidence by itself, the two states live on different distributions, means near 0.57 and 0.28, so the comparable quantities are the contributions beta times state, roughly 0.09 conceptual against 0.02 procedural in a matched-initialization rerun, an ordering that is stable across seeds. Conceptual contributes more, but neither contribution improves held-out prediction, and no sum of these betas is comparable to notebook 05's pooled beta, the features and combination rules differ.

**The rest of the grid behaves as the designed stress cases.** With both branches open the mastered betas collapse toward zero while the guess-branch pair absorbs variance on a clipped, weakly identified likelihood, a warning against reading any both-row beta as meaningful. Mastered beta-equals-one repeats the F1 threshold effect, f1 up with AUC held, and the correction from notebook 05 applies here too, the lift comes from predicting more correct turns after the refit compensates, not from predicting more incorrect ones. Both-pinned subtracts up to two per branch and collapses as designed.

**Conclusion.** Grouping further weakens an already redundant channel, merging partially observed family annotations into broad conceptual and procedural states whose pre-turn values barely predict correctness at all. The conceptual group carries more of what little contribution exists, at contribution level and stably across seeds, but neither group improves held-out ranking, and the fitted values are read as descriptive under the same selection and initialization caveats as notebook 05.

## 8. Notes

- Every model keeps its fitted parameters at `.parameters` and the shared frozen chains at `.chains`, nothing here refits the chains.
- The chains' two columns arrive in GROUPS order, conceptual then procedural, and the strict compact-position check raises on any frame and chain-sequence disagreement rather than realigning silently.
- The winner's row name and betas get logged with the pick, and the F3 face reuses this notebook's shape with five per-family channels.